## Step 1: Raw Data Mapping

This notebook reads the NHANES raw `.xpt` files, maps every variable to its question text andevery numeric code to its human-readable meaning, then saves everything to one `.xlsx`.

#### Import Libraries

In [1]:
#pip install pyreadstat openpyxl xlsxwriter

In [2]:
import glob
import os
import re
import pandas as pd
import pyreadstat

pd.set_option("display.width", 160)

#### Data Path and Cycle Definitions

In [3]:
BASE_DIR    = "."
RAW_DIR     = os.path.join(BASE_DIR, "Nhanes_RawData")
MAPPED_XLSX = os.path.join(RAW_DIR, "Nhanes_Mapped.xlsx")
MERGED_CSV  = os.path.join(RAW_DIR, "Nhanes_Merged_Raw.csv")   # fast re-load for Step 2

# suffix -> (short label, human label, years of data the weight represents)
CYCLES = {
    "P": ("2017-Mar2020", "NHANES 2017 - March 2020 (pre-pandemic)", 3.2),
    "L": ("2021-2023",    "NHANES August 2021 - August 2023",        2.0),
}

# file prefix -> friendly topic label (suffix-independent)
FILE_TOPICS = {
    "DEMO": "Demographics",
    "DPQ" : "Depression (PHQ-9)",
    "ALQ" : "Alcohol Use",
    "SMQ" : "Smoking",
    "SLQ" : "Sleep Disorders",
    "PAQ" : "Physical Activity",
    "HSQ" : "Health Status",
    "INQ" : "Income",
}

# weight columns that mean the same thing but are named differently per cycle
WEIGHT_ALIASES = {
    "WTINT": ["WTINT2YR", "WTINTPRP"],   # interview weight
    "WTMEC": ["WTMEC2YR", "WTMECPRP"],   # MEC exam weight
}

#### Read every .xpt in the raw folderFilenames look like `DPQ_L.xpt` or `DPQ_P.xpt`. We split the name into `prefix` + `suffix`so the same code handles both cycles.

In [4]:
def read_xpt(path):
    """Read a SAS transport file. Try utf-8, fall back to latin1."""
    for enc in ("utf-8", "latin1"):
        try:
            return pyreadstat.read_xport(path, encoding=enc)
        except Exception:
            continue
    raise RuntimeError(f"Could not read {path}")


def split_name(stem):
    """'DPQ_L' -> ('DPQ', 'L').  Unknown pattern -> (stem, '?')."""
    m = re.fullmatch(r"([A-Za-z]+)_([A-Za-z])", stem)
    return (m.group(1).upper(), m.group(2).upper()) if m else (stem.upper(), "?")


data, metas = {}, {}          # key = 'DPQ_L'
found = {suf: [] for suf in CYCLES}

for path in sorted(glob.glob(os.path.join(RAW_DIR, "*.xpt"))):
    stem = os.path.splitext(os.path.basename(path))[0]
    prefix, suffix = split_name(stem)
    if suffix not in CYCLES:
        print(f"  !! skipping {stem} - unknown cycle suffix '{suffix}'")
        continue
    df, meta = read_xpt(path)
    data[stem], metas[stem] = df, meta
    found[suffix].append(prefix)
    print(f"loaded {stem:8s} cycle={CYCLES[suffix][0]:12s} shape={df.shape}")

print()
for suf, (lbl, _, _) in CYCLES.items():
    print(f"{lbl:14s} topics: {sorted(found[suf])}")

# warn about topics that exist in one cycle but not the other
only_p = set(found["P"]) - set(found["L"])
only_l = set(found["L"]) - set(found["P"])
if only_p: print(f"\n  !! only in _P: {sorted(only_p)}")
if only_l: print(f"  !! only in _L: {sorted(only_l)}")

loaded ALQ_L    cycle=2021-2023    shape=(6337, 9)
loaded ALQ_P    cycle=2017-Mar2020 shape=(8965, 10)
loaded DEMO_L   cycle=2021-2023    shape=(11933, 27)
loaded DEMO_P   cycle=2017-Mar2020 shape=(15560, 29)
loaded DPQ_L    cycle=2021-2023    shape=(6337, 11)
loaded DPQ_P    cycle=2017-Mar2020 shape=(8965, 11)
loaded HSQ_L    cycle=2021-2023    shape=(6615, 2)
loaded HSQ_P    cycle=2017-Mar2020 shape=(9445, 2)
loaded INQ_L    cycle=2021-2023    shape=(11933, 5)
loaded INQ_P    cycle=2017-Mar2020 shape=(15560, 3)
loaded PAQ_L    cycle=2021-2023    shape=(8153, 8)
loaded PAQ_P    cycle=2017-Mar2020 shape=(9693, 17)
loaded SLQ_L    cycle=2021-2023    shape=(8501, 7)
loaded SLQ_P    cycle=2017-Mar2020 shape=(10195, 11)
loaded SMQ_L    cycle=2021-2023    shape=(9015, 9)
loaded SMQ_P    cycle=2017-Mar2020 shape=(11137, 16)

2017-Mar2020   topics: ['ALQ', 'DEMO', 'DPQ', 'HSQ', 'INQ', 'PAQ', 'SLQ', 'SMQ']
2021-2023      topics: ['ALQ', 'DEMO', 'DPQ', 'HSQ', 'INQ', 'PAQ', 'SLQ', 'SMQ']


#### The value mapping transcribed from the NHANES codebooks at <https://wwwn.cdc.gov/nchs/nhanes>.The `.xpt` file gives us the question TEXT for free (via `meta.column_names_to_labels`);only these `code -> meaning` pairs must be added by hand.Codes are shared across the two cycles unless noted. `_P only` marks variables that existin the 2017-Mar 2020 files but not in the 2021-2023 files.

In [5]:
YESNO  = {1: "Yes", 2: "No", 7: "Refused", 9: "Don't know"}
YESNO2 = {1: "Yes", 2: "No", 77: "Refused", 99: "Don't know"}
PHQ    = {0: "Not at all", 1: "Several days", 2: "More than half the days",
          3: "Nearly every day", 7: "Refused", 9: "Don't know"}
DAYS7  = {**{d: f"{d} day(s)" for d in range(1, 8)}, 77: "Refused", 99: "Don't know"}
FREQ_NIGHTS = {0: "Never", 1: "Rarely - 1-2 nights a week",
               2: "Occasionally - 3-4 nights a week",
               3: "Frequently - 5 or more nights a week",
               7: "Refused", 9: "Don't know"}
DRINK_FREQ = {0: "Never in the last year", 1: "Every day", 2: "Nearly every day",
              3: "3 to 4 times a week", 4: "2 times a week", 5: "Once a week",
              6: "2 to 3 times a month", 7: "Once a month",
              8: "7 to 11 times in the last year", 9: "3 to 6 times in the last year",
              10: "1 to 2 times in the last year", 77: "Refused", 99: "Don't know"}
BINGE_FREQ = {0: "Never", 1: "Every day", 2: "Nearly every day",
              3: "3 to 4 times a week", 4: "2 times a week", 5: "Once a week",
              6: "2 to 3 times a month", 7: "Once a month",
              8: "7 to 11 times in the last year", 9: "3 to 6 times in the last year",
              10: "1 to 2 times in the last year", 77: "Refused", 99: "Don't know"}

VALUE_LABELS = {
    # ---------- DPQ : PHQ-9 depression (target) - identical in _P and _L ----------
    "DPQ010": PHQ, "DPQ020": PHQ, "DPQ030": PHQ, "DPQ040": PHQ, "DPQ050": PHQ,
    "DPQ060": PHQ, "DPQ070": PHQ, "DPQ080": PHQ, "DPQ090": PHQ,
    "DPQ100": {0: "Not at all difficult", 1: "Somewhat difficult",
               2: "Very difficult", 3: "Extremely difficult",
               7: "Refused", 9: "Don't know"},

    # ---------- DEMO : demographics ----------
    "RIDSTATR": {1: "Interviewed only", 2: "Interviewed + MEC examined"},
    "RIAGENDR": {1: "Male", 2: "Female"},
    "RIDRETH1": {1: "Mexican American", 2: "Other Hispanic", 3: "Non-Hispanic White",
                 4: "Non-Hispanic Black", 5: "Other Race - Incl Multi-Racial"},
    "RIDRETH3": {1: "Mexican American", 2: "Other Hispanic", 3: "Non-Hispanic White",
                 4: "Non-Hispanic Black", 6: "Non-Hispanic Asian",
                 7: "Other Race - Incl Multi-Racial"},
    "RIDEXMON": {1: "Nov 1 - Apr 30", 2: "May 1 - Oct 31"},
    "SDDSRVYR": {12: "NHANES Aug 2021 - Aug 2023", 66: "NHANES 2017 - Mar 2020"},
    "DMQMILIZ": YESNO,                                    # _L only
    "DMDBORN4": {1: "Born in 50 US States or DC", 2: "Others",
                 77: "Refused", 99: "Don't know"},
    "DMDEDUC2": {1: "Less than 9th grade", 2: "9-11th grade (no diploma)",
                 3: "High school grad / GED", 4: "Some college or AA degree",
                 5: "College graduate or above", 7: "Refused", 9: "Don't know"},
    "DMDMARTZ": {1: "Married / Living with partner",
                 2: "Widowed / Divorced / Separated", 3: "Never married",
                 77: "Refused", 99: "Don't know"},
    "RIDEXPRG": {1: "Yes, positive lab pregnancy test", 2: "Not pregnant",
                 3: "Cannot ascertain"},
    "DMDHRGND": {1: "Male", 2: "Female"},                 # _L only
    "DMDHRAGZ": {1: "< 20 years", 2: "20-39 years", 3: "40-59 years",
                 4: "60+ years"},                          # _L only
    "DMDHREDZ": {1: "Less than high school degree",
                 2: "High school grad/GED or some college/AA degree",
                 3: "College graduate or above",
                 7: "Refused", 9: "Don't know"},           # _L only
    "DMDHSEDZ": {1: "Less than high school degree",
                 2: "High school grad/GED or some college/AA degree",
                 3: "College graduate or above",
                 7: "Refused", 9: "Don't know"},           # _L only
    "DMDHRMAZ": {1: "Married / Living with partner",
                 2: "Widowed / Divorced / Separated", 3: "Never married",
                 77: "Refused", 99: "Don't know"},         # _L only
    "DMDYRUSR": {1: "Less than 5 years", 2: "5 years to less than 15 years",
                 3: "15 years to less than 30 years", 4: "30 years or more",
                 77: "Refused", 99: "Don't know"},         # _L only
    "DMDYRUSZ": {1: "Less than 5 years", 2: "5 years to less than 15 years",
                 3: "15 years to less than 30 years", 4: "30 years or more",
                 77: "Refused", 99: "Don't know"},         # _P only
    # _P only interview-mode / language flags
    "AIALANGA": {1: "English", 2: "Spanish", 3: "Asian languages",
                 4: "Other language"},
    "FIAINTRP": YESNO, "MIAINTRP": YESNO, "SIAINTRP": YESNO,
    "FIAPROXY": YESNO, "MIAPROXY": YESNO, "SIAPROXY": YESNO,
    "FIALANG":  {1: "English", 2: "Spanish", 3: "Asian languages", 4: "Other"},
    "MIALANG":  {1: "English", 2: "Spanish", 3: "Asian languages", 4: "Other"},
    "SIALANG":  {1: "English", 2: "Spanish", 3: "Asian languages", 4: "Other"},

    # ---------- ALQ : alcohol ----------
    "ALQ111": YESNO,
    "ALQ121": DRINK_FREQ,
    "ALQ142": BINGE_FREQ,
    "ALQ270": BINGE_FREQ,
    "ALQ280": BINGE_FREQ,
    "ALQ290": BINGE_FREQ,                                  # _P only
    "ALQ151": YESNO,

    # ---------- SMQ : smoking ----------
    "SMQ020": YESNO,
    "SMQ040": {1: "Every day", 2: "Some days", 3: "Not at all",
               7: "Refused", 9: "Don't know"},
    "SMQ050U": {1: "Days", 2: "Weeks", 3: "Months", 4: "Years",
                7: "Refused", 9: "Don't know"},            # _P only
    "SMQ078": {1: "Within 5 minutes", 2: "From 6 to 30 minutes",
               3: "From more than 30 minutes to one hour",
               4: "From more than 1 hour to 2 hours",
               5: "From more than 2 hours to 3 hours",
               6: "From more than 3 hours to 4 hours",
               7: "More than 4 hours", 77: "Refused", 99: "Don't know"},  # _P only
    "SMD100FL": {0: "Non-filtered", 1: "Filtered", 7: "Refused",
                 9: "Don't know"},                          # _P only
    "SMD100MN": {0: "Non-menthol", 1: "Menthol", 7: "Refused", 9: "Don't know"},
    "SMQ670": YESNO,                                        # _P only
    "SMQ621": {1: "I have never smoked, not even a puff",
               2: "1 or more puffs but never a whole cigarette",
               3: "1 cigarette", 4: "2 to 5 cigarettes", 5: "6 to 15 cigarettes",
               6: "16 to 25 cigarettes", 7: "26 to 99 cigarettes",
               8: "100 or more cigarettes", 77: "Refused", 99: "Don't know"},
    "SMAQUEX2": {1: "Home Interview / CAPI (18+ yrs)", 2: "ACASI (12-17 yrs)"},

    # ---------- SLQ : sleep disorders ----------
    "SLQ030": FREQ_NIGHTS,                                  # _P only - snoring
    "SLQ040": FREQ_NIGHTS,                                  # _P only - snort/stop breathing
    "SLQ050": YESNO,                                        # _P only - told doctor
    "SLQ120": {0: "Never", 1: "Rarely - 1 time a month",
               2: "Sometimes - 2-4 times a month",
               3: "Often - 5-15 times a month",
               4: "Almost always - 16-30 times a month",
               7: "Refused", 9: "Don't know"},              # _P only - daytime sleepiness

    # ---------- PAQ : physical activity ----------
    # _L short block (letter-coded units)
    "PAD790U": {"D": "Day", "W": "Week", "M": "Month", "Y": "Year"},
    "PAD810U": {"D": "Day", "W": "Week", "M": "Month", "Y": "Year"},
    # _P GPAQ block
    "PAQ605": YESNO, "PAQ620": YESNO, "PAQ635": YESNO,
    "PAQ650": YESNO, "PAQ665": YESNO,
    "PAQ610": DAYS7, "PAQ625": DAYS7, "PAQ640": DAYS7,
    "PAQ655": DAYS7, "PAQ670": DAYS7,

    # ---------- HSQ : health status ----------
    "HSQ590": YESNO,

    # ---------- INQ : income ----------
    "INDFMMPC": {1: "Index <= 1.30", 2: "1.30 < Index <= 1.85", 3: "Index > 1.85",
                 7: "Refused", 9: "Don't know"},
    "INQ300": YESNO,                                        # _L only
    "IND310": {1: "$0 to $500", 2: "$501 to $1,000", 3: "$1,001 to $2,000",
               4: "$2,001 to $3,000", 5: "$3,001 to $5,000",
               6: "$5,001 to $10,000", 7: "More than $10,000",
               77: "Refused", 99: "Don't know"},            # _L only
}

print(f"value-labelled variables: {len(VALUE_LABELS)}")

value-labelled variables: 74


#### Sentinel "missing" codes on continuous variablesThese are NOT real numbers. If Step 2 averages them the results are garbage(e.g. `PAD680 = 9999` means "Don't know", not 9,999 minutes of sitting).We record them here so the codebook carries them forward.

In [6]:
MISSING_CODES = {
    # alcohol
    "ALQ130": {777: "Refused", 999: "Don't know"},
    "ALQ170": {777: "Refused", 999: "Don't know"},
    # smoking
    "SMD030": {0: "Never smoked cigarettes regularly", 777: "Refused", 999: "Don't know"},
    "SMD057": {1: "1 cigarette or less", 95: "95 cigarettes or more",
               777: "Refused", 999: "Don't know"},
    "SMD650": {1: "1 cigarette or less", 95: "95 cigarettes or more",
               777: "Refused", 999: "Don't know"},
    "SMD641": {77: "Refused", 99: "Don't know"},
    "SMD630": {6: "6 years or less", 77: "Refused", 99: "Don't know"},
    "SMQ050Q": {66666: "50 or more years", 77777: "Refused", 99999: "Don't know"},
    # sleep
    "SLD012": {2: "Less than 3 hours", 14: "14 hours or more"},
    "SLD013": {2: "Less than 3 hours", 14: "14 hours or more"},
    "SLQ300": {"77777": "Refused", "99999": "Don't know"},
    "SLQ310": {"77777": "Refused", "99999": "Don't know"},
    "SLQ320": {"77777": "Refused", "99999": "Don't know"},
    "SLQ330": {"77777": "Refused", "99999": "Don't know"},
    # physical activity (minutes)
    "PAD615": {7777: "Refused", 9999: "Don't know"},
    "PAD630": {7777: "Refused", 9999: "Don't know"},
    "PAD645": {7777: "Refused", 9999: "Don't know"},
    "PAD660": {7777: "Refused", 9999: "Don't know"},
    "PAD675": {7777: "Refused", 9999: "Don't know"},
    "PAD680": {7777: "Refused", 9999: "Don't know"},
    "PAD800": {7777: "Refused", 9999: "Don't know"},
    "PAD820": {7777: "Refused", 9999: "Don't know"},
    "PAD790Q": {7777: "Refused", 9999: "Don't know"},
    "PAD810Q": {7777: "Refused", 9999: "Don't know"},
    # demographics / income
    "RIDAGEMN": {},
    "INDFMPIR": {},
    "DMDHHSIZ": {},
}
# drop the empty placeholders
MISSING_CODES = {k: v for k, v in MISSING_CODES.items() if v}
print(f"variables with sentinel codes: {len(MISSING_CODES)}")

variables with sentinel codes: 24


#### Decoding the cells

In [7]:
def decode(df):
    """Replace numeric codes with their human-readable meaning where we have a mapping."""
    out = df.copy()
    for col in out.columns:
        if col in VALUE_LABELS:
            m = VALUE_LABELS[col]
            out[col] = out[col].map(lambda v, m=m: m.get(v, v))   # keep value if unmapped
    return out

#### Merge inside each cycle, then stack the cycles`SEQN` is unique **within** a cycle only, so the join happens per cycle and the two results arestacked row-wise. `CYCLE` / `CYCLE_LABEL` / `SOURCE_SUFFIX` tell you where each row came from.

In [8]:
def harmonize_weights(df, suffix):
    """Give the cycle-specific weight columns neutral names, keeping the originals."""
    for neutral, aliases in WEIGHT_ALIASES.items():
        for a in aliases:
            if a in df.columns:
                df[neutral] = df[a]
                break
    return df


cycle_frames = {}
for suffix, (short, long_label, years) in CYCLES.items():
    frames = {p: data[f"{p}_{suffix}"] for p in found[suffix]}
    if not frames:
        print(f"  !! no files for cycle {short}, skipped")
        continue

    # anchor on DEMO so every respondent in the cycle survives the join
    order  = (["DEMO"] if "DEMO" in frames else []) + [p for p in frames if p != "DEMO"]
    merged = None
    for p in order:
        merged = frames[p] if merged is None else merged.merge(frames[p], on="SEQN", how="outer")

    merged = harmonize_weights(merged, suffix)
    merged.insert(1, "CYCLE",         short)
    merged.insert(2, "CYCLE_LABEL",   long_label)
    merged.insert(3, "SOURCE_SUFFIX", suffix)
    cycle_frames[suffix] = merged
    print(f"{short:12s} merged shape={merged.shape}  topics={order}")

merged_all = pd.concat(cycle_frames.values(), ignore_index=True, sort=False)
print(f"\nSTACKED shape = {merged_all.shape}")

2017-Mar2020 merged shape=(15560, 97)  topics=['DEMO', 'ALQ', 'DPQ', 'HSQ', 'INQ', 'PAQ', 'SLQ', 'SMQ']
2021-2023    merged shape=(11933, 76)  topics=['DEMO', 'ALQ', 'DPQ', 'HSQ', 'INQ', 'PAQ', 'SLQ', 'SMQ']

STACKED shape = (27493, 116)


#### Pooled survey weightsNHANES weights are scaled to the number of years each cycle represents:`_P` covers 3.2 years, `_L` covers 2 years, so the pooled window is 5.2 years.```WTINT_COMB = WTINT * (years_of_this_cycle / 5.2)```> **Note for the write-up:** NCHS publishes official guidance for combining *2-year* cycles and for> the pre-pandemic file on its own. Combining 2017-Mar 2020 with 2021-2023 is an analyst decision,> not an NCHS-blessed recipe. State this as a limitation, and run at least one sensitivity check> on the `_L` cycle alone before you trust a pooled prevalence estimate.

In [9]:
TOTAL_YEARS = sum(y for _, _, y in CYCLES.values())

for neutral in WEIGHT_ALIASES:
    comb = f"{neutral}_COMB"
    merged_all[comb] = pd.NA
    for suffix, (short, _, years) in CYCLES.items():
        mask = merged_all["SOURCE_SUFFIX"] == suffix
        if neutral in merged_all.columns:
            merged_all.loc[mask, comb] = merged_all.loc[mask, neutral] * (years / TOTAL_YEARS)
    merged_all[comb] = pd.to_numeric(merged_all[comb], errors="coerce")

print(f"pooled window = {TOTAL_YEARS} years")
for neutral in WEIGHT_ALIASES:
    comb = f"{neutral}_COMB"
    if comb in merged_all:
        print(f"  sum({comb}) = {merged_all[comb].sum():,.0f}  "
              f"(non-null n={merged_all[comb].notna().sum():,})")

pooled window = 5.2 years
  sum(WTINT_COMB) = 324,127,779  (non-null n=27,493)
  sum(WTMEC_COMB) = 324,127,779  (non-null n=27,493)


#### Build the codebookOne row per `variable x code`. The `Available_In` column tells you whether a variable exists inboth cycles, only in the pre-pandemic files, or only in the current files - this is what you willuse in Step 2 to decide what to keep.

In [10]:
# which cycles does each variable appear in?
var_cycles = {}
for stem, df in data.items():
    prefix, suffix = split_name(stem)
    for col in df.columns:
        var_cycles.setdefault(col, set()).add(suffix)

def availability(col):
    s = var_cycles.get(col, set())
    if s == set(CYCLES):        return "both"
    if s == {"P"}:              return "_P only (2017-Mar2020)"
    if s == {"L"}:              return "_L only (2021-2023)"
    return "derived / added"

codebook, decoded_frames = [], {}

for stem, df in sorted(data.items()):
    prefix, suffix = split_name(stem)
    topic  = FILE_TOPICS.get(prefix, prefix)
    meta   = metas[stem]
    cyc    = CYCLES[suffix][0]

    for col in df.columns:
        q    = meta.column_names_to_labels.get(col, "")
        avail = availability(col)
        rows = []
        if col in VALUE_LABELS:
            rows = [(code_, meaning, "categorical")
                    for code_, meaning in VALUE_LABELS[col].items()]
        elif col in MISSING_CODES:
            rows = [("(all other values)", "continuous / numeric", "continuous")]
            rows += [(code_, meaning, "special code")
                     for code_, meaning in MISSING_CODES[col].items()]
        else:
            kind = "respondent ID" if col == "SEQN" else "continuous / numeric"
            rows = [("(all values)", kind, "continuous")]

        for code_, meaning, kind in rows:
            codebook.append([topic, cyc, stem, col, q, avail, kind, code_, meaning])

    sheet = f"{topic} {cyc}"[:31]
    decoded_frames[sheet] = decode(df)

codebook_df = pd.DataFrame(codebook, columns=[
    "Topic", "Cycle", "File", "Variable", "Question",
    "Available_In", "Type", "Code", "Meaning"])

print(f"codebook rows = {len(codebook_df):,}")
print()
print(codebook_df.drop_duplicates("Variable")["Available_In"].value_counts())

codebook rows = 758

Available_In
both                      52
_P only (2017-Mar2020)    40
_L only (2021-2023)       19
Name: count, dtype: int64


#### Save one .xlsx (plus a CSV for fast re-loading in Step 2)The CSV is written **first** because it is what Step 2 actually reads - it holds the raw numericcodes, loads in under a second, and never truncates. The Excel file is the human-readabledeliverable; writing ~27k rows across a dozen sheets takes a minute or two, so be patient.

In [11]:
import time

# 1) fast, lossless artifact for Step 2 - raw codes, no decoding
merged_all.to_csv(MERGED_CSV, index=False)
print(f"Saved {MERGED_CSV}  {merged_all.shape}")

# 2) human-readable workbook
merged_decoded = decode(merged_all)
engine = "xlsxwriter"
try:
    import xlsxwriter          # noqa: F401  (roughly 2x faster than openpyxl)
except ImportError:
    engine = "openpyxl"

t0 = time.time()
with pd.ExcelWriter(MAPPED_XLSX, engine=engine) as xl:
    codebook_df.to_excel(xl, sheet_name="Codebook", index=False)
    for sheet, dec in decoded_frames.items():
        dec.to_excel(xl, sheet_name=sheet, index=False)
    for suffix, dfm in cycle_frames.items():
        dfm.to_excel(xl, sheet_name=f"Merged_{CYCLES[suffix][0]}"[:31], index=False)
    merged_decoded.to_excel(xl, sheet_name="Merged_Decoded", index=False)

print(f"Saved {MAPPED_XLSX}  (engine={engine}, {time.time()-t0:.0f}s)")
print(f"      codebook rows  = {len(codebook_df):,}")
print(f"      stacked merged = {merged_all.shape}")

Saved .\Nhanes_RawData\Nhanes_Merged_Raw.csv  (27493, 118)
Saved .\Nhanes_RawData\Nhanes_Mapped.xlsx  (engine=openpyxl, 68s)
      codebook rows  = 758
      stacked merged = (27493, 118)


#### Quality checksRun these every time. If any line prints `FAIL`, stop and fix before moving to Step 2.

In [12]:
def check(label, ok, detail=""):
    print(f"[{'PASS' if ok else 'FAIL'}] {label}" + (f"  -> {detail}" if detail else ""))

# 1. no SEQN collision between the two cycles
seqn_sets = {s: set(df["SEQN"]) for s, df in cycle_frames.items()}
overlap = set.intersection(*seqn_sets.values()) if len(seqn_sets) > 1 else set()
check("SEQN does not overlap across cycles", len(overlap) == 0, f"{len(overlap)} shared IDs")

# 2. row count = sum of per-cycle rows
expected = sum(len(df) for df in cycle_frames.values())
check("stacked rows == sum of cycle rows", len(merged_all) == expected,
      f"{len(merged_all):,} vs {expected:,}")

# 3. no duplicate SEQN within a cycle
for s, df in cycle_frames.items():
    dup = df["SEQN"].duplicated().sum()
    check(f"no duplicate SEQN in {CYCLES[s][0]}", dup == 0, f"{dup} duplicates")

# 4. CYCLE agrees with the SDDSRVYR value NHANES stamps on each file
if "SDDSRVYR" in merged_all.columns:
    xt = merged_all.groupby("CYCLE")["SDDSRVYR"].nunique()
    check("one SDDSRVYR per cycle", (xt == 1).all(), xt.to_dict())

# 5. PHQ-9 present for both cycles
dpq = [c for c in merged_all.columns if c.startswith("DPQ0")]
cov = merged_all.groupby("CYCLE")[dpq[0]].apply(lambda s: s.notna().sum()) if dpq else None
check("PHQ-9 items present in both cycles", cov is not None and (cov > 0).all(),
      cov.to_dict() if cov is not None else "no DPQ columns")

# 6. weights present for every row
for neutral in WEIGHT_ALIASES:
    if neutral in merged_all.columns:
        miss = merged_all[neutral].isna().sum()
        check(f"{neutral} present on all rows", miss == 0, f"{miss:,} missing")

print()
print("Coverage of key columns by cycle (non-null counts):")
key = [c for c in ["RIDAGEYR", "RIAGENDR", "DPQ010", "SLD012", "SLQ050",
                   "PAD680", "PAQ650", "PAD810Q", "ALQ121", "SMQ020",
                   "HSQ590", "INDFMPIR"] if c in merged_all.columns]
print(merged_all.groupby("CYCLE")[key].count().T)

[PASS] SEQN does not overlap across cycles  -> 0 shared IDs
[PASS] stacked rows == sum of cycle rows  -> 27,493 vs 27,493
[PASS] no duplicate SEQN in 2017-Mar2020  -> 0 duplicates
[PASS] no duplicate SEQN in 2021-2023  -> 0 duplicates
[PASS] one SDDSRVYR per cycle  -> {'2017-Mar2020': 1, '2021-2023': 1}
[PASS] PHQ-9 items present in both cycles  -> {'2017-Mar2020': 8308, '2021-2023': 5519}
[PASS] WTINT present on all rows  -> 0 missing
[PASS] WTMEC present on all rows  -> 0 missing

Coverage of key columns by cycle (non-null counts):
CYCLE     2017-Mar2020  2021-2023
RIDAGEYR         15560      11933
RIAGENDR         15560      11933
DPQ010            8308       5519
SLD012           10105       8388
SLQ050           10195          0
PAD680            9676       8138
PAQ650            9693          0
PAD810Q              0       8139
ALQ121            7503       4922
SMQ020            9693       8135
HSQ590            8841       5751
INDFMPIR         13359       9892
